# EDGAR Financial Sentiment — Part 5: LLM-as-Classifier (Zero-/Few-Shot)

**Track B begins.** Parts 2–4 all *trained* something. Here we train **nothing**: we just **prompt** `mistralai/Mistral-7B-Instruct-v0.3` (4-bit, same model as Part 4) to read a sentence and name its sentiment. Zero-shot first (instruction only), then few-shot (a few labeled examples in the prompt). Scored on the same Financial PhraseBank test set, this is the **no-training baseline** the whole project is measured against — if prompting gets close to your fine-tuned models, that changes the build-vs-prompt calculus (see the use-case lab).

**What you'll practice — a new skill area:** not training loops this time, but the *prompting* workflow — **(1)** constructing the classification prompt, **(2)** parsing the model's free-text reply into a label, and **(3)** the inference/eval loop that ties it together. Those three are your blanks.

> **Run in Google Colab with a T4 GPU.** Uses the same gated Mistral model as Part 4.

### The idea: classification as instruction-following

An instruction-tuned LLM will answer a well-posed question. So instead of a trained classifier head, we ask:

> *"Classify this financial sentence as negative, neutral, or positive. Reply with one word."*

and read the answer out of the generated text. **Zero-shot** = instruction only. **Few-shot** = paste a few labeled examples first, which both demonstrates the output format and calibrates the model. The two hard parts in practice are writing a prompt the model follows *reliably*, and **parsing** a free-text reply (`'Positive.'`, `'The sentiment is positive'`, `'I'd say mixed'`) back into a clean label.

## 0. Setup

In [ ]:
!pip install -q -U transformers bitsandbytes accelerate datasets

In [ ]:
import random, io, zipfile, requests
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import Dataset as HFDataset, ClassLabel

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert device.type == 'cuda', 'Switch the Colab runtime to a T4 GPU before running.'

### Hugging Face login (Mistral is gated)
Same as Part 4: accept the license at <https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3>, make a token, and log in below.

In [ ]:
from huggingface_hub import login
login()   # paste your HF token when prompted

## 1. Config & seeds

In [ ]:
torch.manual_seed(10); random.seed(10); np.random.seed(10)

MODEL_ID       = 'mistralai/Mistral-7B-Instruct-v0.3'
EVAL_N         = 150     # subsample of the test set (generation is slow); raise toward 453 for the full set
MAX_NEW_TOKENS = 8       # we only need one word back

LABELS       = ['negative', 'neutral', 'positive']   # the allowed answers (index 0/1/2)
LABEL_TO_IDX = {name: i for i, name in enumerate(LABELS)}

## 2. Load PhraseBank, subsample the test set, pick few-shot examples

Same data + stratified split as before. We evaluate on a ~`EVAL_N` slice (generation per example is slow), and pull **one few-shot example per class from the *train* split** — never the test set, to avoid leakage.

In [ ]:
_URL = 'https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip'
_z = zipfile.ZipFile(io.BytesIO(requests.get(_URL, timeout=120).content))
_member = next(n for n in _z.namelist() if n.endswith('Sentences_AllAgree.txt'))
_raw = _z.read(_member).decode('latin-1')

_label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
_rows = []
for _line in _raw.splitlines():
    _line = _line.strip()
    if not _line:
        continue
    _sent, _lab = _line.rsplit('@', 1)
    _rows.append({'sentence': _sent, 'label': _label_map[_lab.strip()]})
_df = pd.DataFrame(_rows)

pb_full = HFDataset.from_pandas(_df).cast_column('label', ClassLabel(names=LABELS))
pb = pb_full.train_test_split(test_size=0.2, seed=10, stratify_by_column='label')
pb_train, pb_test = pb['train'], pb['test']

# evaluation subsample (shuffled then truncated; the split is already class-balanced)
pb_eval = pb_test.shuffle(seed=10).select(range(min(EVAL_N, len(pb_test))))

# one few-shot example per class, from TRAIN
FEWSHOT_EXAMPLES = []
for _cls in range(3):
    _ex = next(e for e in pb_train if e['label'] == _cls)
    FEWSHOT_EXAMPLES.append((_ex['sentence'], LABELS[_cls]))

print('Eval subsample:', len(pb_eval), 'sentences')
print('Few-shot examples (one per class):')
for s, lab in FEWSHOT_EXAMPLES:
    print(f'  [{lab}] {s[:80]}...')

## 3. Load Mistral-7B (4-bit) + a generation helper  *(provided)*

We load the model as a **causal LM** (a text generator), not a classifier head — there's no training. `generate_reply` is the transformers plumbing: it wraps your prompt in the chat template, generates a few tokens **greedily** (deterministic), and returns just the newly generated text.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map={'': 0})
model.eval()
print('Loaded', MODEL_ID)

In [ ]:
@torch.no_grad()
def generate_reply(prompt_text, max_new_tokens=MAX_NEW_TOKENS):
    """Send one user message through the chat template and return the model's reply text."""
    messages = [{'role': 'user', 'content': prompt_text}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors='pt').to(device)
    out = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    new_tokens = out[0][inputs.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 4. Build the prompt  &larr; **your code**

Write `build_prompt(sentence, examples=())` returning the **user-message string**. It should:
1. give an **instruction** naming the three allowed labels and asking for **one word only**;
2. include any **few-shot examples** (each an `(sentence, label)` pair) in the same format you'll ask the target in;
3. end with the target sentence and a `Sentiment:` cue for the model to continue.

The *same* function serves both modes: zero-shot passes `examples=()`, few-shot passes the list. Prompt wording matters a lot — you'll iterate on it.

In [ ]:
def build_prompt(sentence, examples=()):
    """Return the user-message text for one classification (zero- or few-shot)."""
    ### YOUR CODE HERE
    lines = [
        'You are a financial-sentiment classifier. Classify each sentence as exactly one of: '
        'negative, neutral, positive. Reply with only that one word.',
        '',
    ]
    for ex_sentence, ex_label in examples:          # few-shot block (empty for zero-shot)
        lines.append(f'Sentence: "{ex_sentence}"')
        lines.append(f'Sentiment: {ex_label}')
        lines.append('')
    lines.append(f'Sentence: "{sentence}"')        # the target
    lines.append('Sentiment:')
    return '\n'.join(lines)
    ### END YOUR CODE

### See your prompt + a real reply
Run after filling in `build_prompt`. Look at the raw reply — that's what your parser (next) has to handle.

In [ ]:
_demo = pb_eval[0]
print('=== PROMPT (zero-shot) ===')
print(build_prompt(_demo['sentence']))
print('\n=== MODEL REPLY ===')
print(repr(generate_reply(build_prompt(_demo['sentence']))))
print('\ntrue label:', LABELS[_demo['label']])

## 5. Parse the reply  &larr; **your code**

`parse_label(reply)` turns the model's free text into a class index `0/1/2`, or `None` if it can't tell. Real replies vary — `'positive'`, `'Positive.'`, `'The sentiment is positive'`. A robust approach: lowercase, then return whichever of the three label words appears **first**. Returning `None` on no-match lets you *count* failures instead of silently guessing.

In [ ]:
def parse_label(reply):
    """Map a free-text reply to a class index 0/1/2, or None if no label word is found."""
    ### YOUR CODE HERE
    text = reply.lower()
    hits = [(text.find(word), idx) for word, idx in LABEL_TO_IDX.items() if word in text]
    if not hits:
        return None
    return min(hits)[1]          # the earliest-appearing label word wins
    ### END YOUR CODE

### Test your parser
It should handle the easy cases and return `None` on the last one.

In [ ]:
for _s in ['positive', 'Positive.', 'The sentiment is negative', 'Neutral', 'I would say mixed']:
    print(f'{_s!r:40s} -> {parse_label(_s)}')

## 6. The evaluation loop  &larr; **your code**

`evaluate(dataset, examples=(), label='zero-shot')` runs the whole pipeline over the eval set: for each sentence, build the prompt, generate a reply, parse it, and tally. Count **unparsed** replies (`pred is None`) separately — a high count means your prompt or parser needs work, not that the model is wrong. (This is the prompting analogue of your Part 2/3 test loop — generation + parsing instead of `argmax`.)

In [ ]:
def evaluate(dataset, examples=(), label='zero-shot'):
    """Classify each sentence via the LLM and report accuracy.

    - for each row: prompt = build_prompt(sentence, examples); reply = generate_reply(prompt);
      pred = parse_label(reply)
    - count correct (pred == true label) and unparsed (pred is None) separately
    - print accuracy + #unparsed; return accuracy %
    """
    ### YOUR CODE HERE
    correct, unparsed, total = 0, 0, 0
    for ex in dataset:
        reply = generate_reply(build_prompt(ex['sentence'], examples))
        pred = parse_label(reply)
        if pred is None:
            unparsed += 1
        elif pred == ex['label']:
            correct += 1
        total += 1
    acc = 100 * correct / max(total, 1)
    print(f'  {label}: accuracy {acc:.1f}%  ({correct}/{total} correct; {unparsed} unparsed)')
    return acc
    ### END YOUR CODE

## 7. Run zero-shot
Instruction only — no examples.

In [ ]:
acc_zero = evaluate(pb_eval, examples=(), label='zero-shot')

## 8. Run few-shot
Same eval set, now with one labeled example per class prepended to every prompt.

In [ ]:
acc_few = evaluate(pb_eval, examples=FEWSHOT_EXAMPLES, label='few-shot')

print(f'\nfew-shot lift over zero-shot: {acc_few - acc_zero:+.1f} pts')

## 9. What to look for

- **Few-shot &ge; zero-shot, usually.** The examples pin down the output format (fewer unparsed replies) and nudge borderline cases. If few-shot *doesn't* help, your zero-shot instruction was probably already clear.
- **The headline comparison:** this no-training baseline vs. your fine-tuned models. Expect prompting to land **below** fine-tuned FinBERT/BERT (~95–97%) — often in the 70s–80s zero-shot, higher few-shot. The real question for the use-case lab is *how big is the gap?* If fine-tuning only buys a few points over a few-shot prompt, prompting may be the right production choice (no labels, no training, instant).
- **Watch the `unparsed` count.** It measures prompt/parser robustness, not model accuracy. If it's high, tighten the instruction ('Reply with exactly one word') before blaming the model.
- **`neutral` is the hard class.** LLMs love to hedge; many errors are neutral-vs-not. Worth inspecting a few misses.

**Plug your numbers into the use-case lab** (`Choosing_a_Training_Method`, Part B) — the zero-shot and few-shot rows of the scoreboard are waiting.

**Things to try:** reword the instruction; add more shots (two per class); ask for a one-word answer *then* a reason (often improves the label); raise `EVAL_N` toward the full 453.

**Next:** Part 6 — **synthetic data generation**: prompt this same model to *generate* labeled financial sentences, add them to the Part 2/3 training set, and measure the accuracy change (Track B feeding Track A).